# Multimodal MIMIC-4 Task Tutorial
**Contributors:** William Pang, Joshua Chen

This notebook demonstrates how to use the `multimodal_mimic4` task.

**Related Resources** (*Note: We might need to update file paths once merged to Sunlab Pyhealth Main*)
- [Multimodal MIMIC4 Example](https://github.com/Multimodal-PyHealth/PyHealth/blob/main/examples/mortality_prediction/multimodal_mimic4.py)
- [Multimodal MIMIC4 Task](https://github.com/Multimodal-PyHealth/PyHealth/blob/main/pyhealth/tasks/multimodal_mimic4.py)

## 0. Environment Setup and Loading Packages

In [1]:
from datetime import datetime
from typing import Any, Dict, List, Optional
import os
import tempfile
import shutil 
import random
import numpy as np
import torch

*Pyhealth Specific Packages*

In [2]:
from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks.multimodal_mimic4 import ClinicalNotesICDLabsMIMIC4

*Set Randomness Seed*

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Notebook-Specific Utility Functions

In [ ]:
def print_patient_admission_info(sample):
    patient = dataset.get_patient(sample['patient_id'])
    admissions = patient.get_events(
        event_type="admissions",
        start=sample['window_start'],
        end=sample['window_end'],
    )
    print(f"\nnum_admissions:  {len(admissions)}")
    for adm in admissions:
        print(f"  hadm_id:       {adm.hadm_id}")
        print(f"  admittime:     {adm.timestamp}")
        print(f"  dischtime:     {adm.dischtime}")

def print_patient_note_info(sample, note_type="discharge", char_limit=80):
    patient = dataset.get_patient(sample['patient_id'])
    notes = patient.get_events(
        event_type=note_type,
        start=sample['window_start'],
        end=sample['window_end'],
    )
    print(f"\n{note_type} notes: {len(notes)}")
    for note in notes:
        print(f"  note_id:   {note.note_id}")
        print(f"  hadm_id:   {note.hadm_id}")
        print(f"  charttime: {note.timestamp}")
        print(f"  storetime: {note.storetime}")
        print(f"  text:      {note.text[:char_limit]}..(Limited to {char_limit} Characters).")

## 2. Load Demo Dataset

We use the MIMIC-IV demo data in `test-resources/core/mimic4demo`. 

This includes:
- Synthetic Notes (`discharge`, `radiology`)

In [5]:
PYHEALTH_REPO_ROOT = '/Users/wpang/Desktop/PyHealth'

EHR_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "test-resources/core/mimic4demo")
NOTE_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "test-resources/core/mimic4demo")
CACHE_DIR = tempfile.mkdtemp()

In [6]:
dataset = MIMIC4Dataset(
    ehr_root=EHR_ROOT,
    note_root=NOTE_ROOT,
    ehr_tables=["diagnoses_icd", "procedures_icd", "labevents"],
    note_tables=["discharge", "radiology"],
    cache_dir=CACHE_DIR,
    num_workers=1,
)

Memory usage Starting MIMIC4Dataset init: 532.1 MB
Initializing mimic4 dataset from /Users/wpang/Desktop/PyHealth/test-resources/core/mimic4demo|/Users/wpang/Desktop/PyHealth/test-resources/core/mimic4demo|None (dev mode: False)
Using provided cache_dir: /var/folders/cr/h1qm2rws041_nzss3n7df3vm0000gn/T/tmpnfe8e_5l/e9840bf9-52d8-575a-afe0-22eed413308e
Initializing MIMIC4EHRDataset with tables: ['diagnoses_icd', 'procedures_icd', 'labevents'] (dev mode: False)
Using default EHR config: /Users/wpang/Desktop/PyHealth/pyhealth/datasets/configs/mimic4_ehr.yaml
Memory usage Before initializing mimic4_ehr: 532.1 MB
Initializing mimic4_ehr dataset from /Users/wpang/Desktop/PyHealth/test-resources/core/mimic4demo (dev mode: False)
Using provided cache_dir: /var/folders/cr/h1qm2rws041_nzss3n7df3vm0000gn/T/tmpnfe8e_5l/e9840bf9-52d8-575a-afe0-22eed413308e/1384e35b-f34a-5dc9-a3fc-c653b0e05a5e
Memory usage After initializing mimic4_ehr: 532.3 MB
Memory usage After EHR dataset initialization: 532.3 MB

/Users/wpang/Desktop/PyHealth/pyhealth/datasets/mimic4.py:121: UserWarning: Events from discharge table only have date timestamp (no specific time). This may affect temporal ordering of events.
  warnings.warn(


## 3. Time Filtering

### 3.1 Full Patient Window
By default the task uses all data between the first admission time and the last discharge time across the patient's processed admissions.

In [7]:
%%capture
task = ClinicalNotesICDLabsMIMIC4()
samples = dataset.set_task(task, num_workers=1)

Setting task ClinicalNotesICDLabsMIMIC4 for mimic4 base dataset...
Task cache paths: task_df=/var/folders/cr/h1qm2rws041_nzss3n7df3vm0000gn/T/tmpnfe8e_5l/e9840bf9-52d8-575a-afe0-22eed413308e/tasks/ClinicalNotesICDLabsMIMIC4_493ab707-d120-581c-a795-319ffd5c468e/task_df.ld, samples=/var/folders/cr/h1qm2rws041_nzss3n7df3vm0000gn/T/tmpnfe8e_5l/e9840bf9-52d8-575a-afe0-22eed413308e/tasks/ClinicalNotesICDLabsMIMIC4_493ab707-d120-581c-a795-319ffd5c468e/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
No cached event dataframe found. Creating: /var/folders/cr/h1qm2rws041_nzss3n7df3vm0000gn/T/tmpnfe8e_5l/e9840bf9-52d8-575a-afe0-22eed413308e/global_event_df.parquet
Combining data from ehr dataset
Scanning table: diagnoses_icd from /Users/wpang/Desktop/PyHealth/test-resources/core/mimic4demo/hosp/diagnoses_icd.csv.gz
Joining with table: /Users/wpang/Desktop/PyHealth/test-resources/core/mimic4demo/hosp/admissions.csv.gz
Scanning table: procedur

In [8]:
random_patient_record = np.random.randint(0, len(samples)-1)
sample = samples[random_patient_record]
sample_patient_id = sample['patient_id']

print(f"\nPatient ID:      {sample_patient_id}")
print(f"Mortality Flag:  {sample['mortality']}")

# Admissions
print_patient_admission_info(sample)

# Discharge Notes
print_patient_note_info(sample, note_type="discharge")

# Radiology Notes
print_patient_note_info(sample, note_type="radiology")


Patient ID:      10009
Mortality Flag:  tensor([0.])
Found 13 unique patient IDs

num_admissions:  1
  hadm_id:       20014
  admittime:     2152-03-20 07:30:00
  dischtime:     2152-03-23 14:00:00

discharge notes: 1
  note_id:   d16
  hadm_id:   20014
  charttime: 2152-03-23 12:00:00
  storetime: 2152-03-23 13:00:00
  text:      Patient 10009 admitted for evaluation. Vitals stable. Labs reviewed. Discharged ..(Limited to 80 Characters).

radiology notes: 1
  note_id:   r16
  hadm_id:   20014
  charttime: 2152-03-20 14:00:00
  storetime: 2152-03-20 15:00:00
  text:      Chest X-ray for patient 10009. No acute cardiopulmonary process identified. Hear..(Limited to 80 Characters).


### 3.2 Random Patient Window

In [9]:
%%capture
task = ClinicalNotesICDLabsMIMIC4(window_hours=12)
samples = dataset.set_task(task, num_workers=1)
sample = samples[random_patient_record]
sample_patient_id = sample['patient_id']

Setting task ClinicalNotesICDLabsMIMIC4 for mimic4 base dataset...
Task cache paths: task_df=/var/folders/cr/h1qm2rws041_nzss3n7df3vm0000gn/T/tmpnfe8e_5l/e9840bf9-52d8-575a-afe0-22eed413308e/tasks/ClinicalNotesICDLabsMIMIC4_6c674c02-3635-50c0-bdde-3a45cb1e6f31/task_df.ld, samples=/var/folders/cr/h1qm2rws041_nzss3n7df3vm0000gn/T/tmpnfe8e_5l/e9840bf9-52d8-575a-afe0-22eed413308e/tasks/ClinicalNotesICDLabsMIMIC4_6c674c02-3635-50c0-bdde-3a45cb1e6f31/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 13 patients. (Polars threads: 16)
Worker 0 finished processing patients.
Fitting processors on the dataset...
Label mortality vocab: {0: 0, 1: 1}
Processing samples and saving to /var/folders/cr/h1qm2rws041_nzss3n7df3vm0000gn/T/tmpnfe8e_5l/e9840bf9-52d8-575a-afe0-22eed413308e/tasks/ClinicalNotesICDLabsMIMIC4_6

In [ ]:
print(f"\nPatient ID:      {sample_patient_id}")
print(f"Mortality Flag:  {sample['mortality']}")
print(f"Window start:    {sample['window_start']}")
print(f"Window end:      {sample['window_end']}")

# Admissions
print_patient_admission_info(sample)

# Discharge Notes
print_patient_note_info(sample, note_type="discharge")

# Radiology Notes
print_patient_note_info(sample, note_type="radiology")

## 4. Cleanup

- Remove `CACHE_DIR`

In [11]:
shutil.rmtree(CACHE_DIR) 